<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>From Scratch MLP - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [54]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [55]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [56]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [57]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [58]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [59]:
internet_required = False

In [60]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

In [61]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [62]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [63]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
22422,49,technician,married,secondary,no,1982,yes,yes,cellular,22,aug,73,15,-1,0,unknown,no
25154,48,management,married,tertiary,no,1600,no,no,cellular,18,nov,448,1,-1,0,unknown,no
42479,54,management,divorced,tertiary,no,380,no,yes,cellular,10,dec,501,1,188,2,success,no
27757,40,technician,single,secondary,no,2017,yes,no,cellular,28,jan,105,1,244,1,other,no
38508,42,entrepreneur,married,secondary,no,169,yes,no,telephone,15,may,87,4,-1,0,unknown,no
25318,30,management,married,secondary,no,1439,yes,no,cellular,18,nov,373,1,-1,0,unknown,no
6288,33,blue-collar,married,secondary,no,0,yes,no,unknown,27,may,190,1,-1,0,unknown,no
1926,41,services,married,secondary,no,510,yes,no,unknown,9,may,231,2,-1,0,unknown,no
15412,53,management,divorced,primary,no,-17,yes,yes,cellular,18,jul,508,1,-1,0,unknown,no
11909,46,management,married,secondary,no,12186,no,no,unknown,20,jun,29,3,-1,0,unknown,no


PCA... feature reduction... not for assessment, but read up.

In [64]:
df.dropna(inplace=True)
df.shape

(45211, 17)

# Conversion and Encoding Ready for the MLP

In [65]:
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Binary columns that should not be one-hot encoded
binary_columns = ['default', 'housing', 'loan', 'y']

# Columns to exclude from the system
excluded_features = ['day', 'campaign', 'pdays', 'previous', 'y']

numerical_to_normalize = [col for col in numerical_cols if col not in binary_columns]
categorical_to_normalize = [col for col in categorical_cols if col not in binary_columns]

### Define Normalisation and Label Encoding Functions

$ X_scaled = \frac{max_X - min_X}{X-min_X}$

In [66]:
def max_min_normalize (df, column_name):
    max_value = df[column_name].max()
    min_value = df[column_name].min()
    df[column_name] = (df[column_name] - min_value) / (max_value - min_value)
    return df

def label_encode(df, column_name):
    # Get unique values in the column
    unique_values = df[column_name].unique()
    
    # Create a mapping of unique values to integers
    mapping = {value: i for i, value in enumerate(unique_values)}
    
    # Apply the mapping to the column
    df[column_name] = df[column_name].map(mapping).astype(float)
    return df

In [67]:
# Clean up the binary columns to be 0 and 1
for col in binary_columns:
    df[col] = df[col].map({'yes': 1.0, 'no': 0.0}).astype(float)

# Normalize the categorical columns using label encoding and then max-min normalization
for col in categorical_to_normalize:
    df = label_encode(df, col)
    df = max_min_normalize(df, col)

# Normalize the numerical columns using max-min normalization
for col in numerical_to_normalize:
    df = max_min_normalize(df, col)


In [68]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
29895,0.181818,0.545455,0.5,0.333333,0.0,0.078959,1.0,0.0,0.5,0.100000,0.727273,0.321065,0.000000,0.283257,0.014545,0.333333,1.0
10922,0.311688,0.272727,0.0,0.666667,0.0,0.077561,0.0,0.0,0.0,0.533333,0.090909,0.044734,0.000000,0.000000,0.000000,0.000000,0.0
39172,0.142857,0.545455,0.5,0.333333,0.0,0.075582,1.0,0.0,0.5,0.566667,0.000000,0.157381,0.048387,0.000000,0.000000,0.000000,1.0
38358,0.350649,0.272727,0.0,1.000000,0.0,0.080974,1.0,0.0,1.0,0.466667,0.000000,0.005287,0.000000,0.391055,0.003636,0.333333,0.0
37282,0.181818,0.000000,0.5,0.333333,0.0,0.076580,1.0,0.0,0.5,0.400000,0.000000,0.058967,0.000000,0.202982,0.007273,0.333333,0.0
29632,0.207792,0.000000,0.0,0.000000,0.0,0.073702,0.0,1.0,0.5,0.066667,0.727273,0.053680,0.016129,0.000000,0.000000,0.000000,0.0
36387,0.168831,0.181818,0.0,0.000000,0.0,0.076662,1.0,0.0,0.5,0.333333,0.000000,0.019520,0.016129,0.000000,0.000000,0.000000,0.0
25028,0.363636,0.090909,0.5,0.333333,0.0,0.088764,1.0,0.0,0.5,0.566667,0.454545,0.022163,0.000000,0.000000,0.000000,0.000000,0.0
25425,0.428571,0.090909,0.0,0.333333,0.0,0.108202,1.0,1.0,1.0,0.566667,0.454545,0.016673,0.080645,0.000000,0.000000,0.000000,0.0
35047,0.441558,0.000000,0.5,0.000000,0.0,0.069771,1.0,0.0,0.5,0.166667,0.000000,0.081741,0.016129,0.000000,0.000000,0.000000,0.0


In [69]:
# sys.exit('Stopping notebook run here.')    

---

### Write out the cleaned dataset for further use



In [70]:
# Create filenames, one timestamped, one 'latest'
filepath = "."
filename = f'{filepath}/bank_full_cleaned_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_cleaned_latest.csv'

# Write out CSV files
df.to_csv(filename, index=False)
df.to_csv(latest_filename, index=False)